In [ ]:
%matplotlib inline
import os
import pandas as pd
import numpy as np
import flopy
import pyemu
import swatmf
import matplotlib.pyplot as plt

### Versions:
swatmf=1.0.1; numpy=2.4.0; pandas=2.3.3; pyemu=1.3.8; flopy=3.9.5

In [ ]:
from swatmf import swatmf_pst_utils, swatmf_pst_par

In [ ]:
swatmf.__version__

# 1. Set up 
## 1.1 write swatmf.con file

In [ ]:
# working directory
prj_dir = r"C:\\Users\\spark\\Documents\\Projects\\Watersheds\\hb_tsu\\opt"
swatmf_model = r"C:\\Users\\spark\\Documents\\Projects\\Watersheds\\hb_tsu\\opt\\SWAT-MODFLOW"
swat_model = r"C:\\Users\\spark\\Documents\\Projects\Watersheds\\hb_tsu\\opt\\SWAT"

In [ ]:
# working directory and file names
# calibration period
sim_start = '1/1/2005'
warmup = 3
cal_start = '1/1/2008'
cal_end = '12/31/2014'

# time step
time_step = 'day'

# extract simulation (what our targets)
# locations
subs = [1]
grids = [5163]

## 1.2 Initiate PEST

In [ ]:
# copy all necessary files (exes) to your working direcotry
swatmf_pst_utils.init_setup(prj_dir, swatmf_model, swat_model)

In [ ]:
swatmf_pst_utils.create_swatmf_con(
    prj_dir, swatmf_model, sim_start, warmup, cal_start, cal_end, subs=subs, grids=grids)

# 2. Build template files

## 2.1 MODFLOW parameters with Pilot Points

In [ ]:
# SET PATH to your optimization working folder
main_opt = "C:\\Users\\spark\\Documents\\Projects\\Watersheds\\hb_tsu\\opt\\main_opt"

### Set up zones for where pilot points will be interpolated

We can have pilot point networks in multiple zones. In this case, we will make a simple zone file using `IDOMAIN` such that all active cells are in the same interpolation zone.

In [ ]:
mname = 'modflow.mfn'
m = flopy.modflow.Modflow.load(
    mname, model_ws=main_opt)

In [ ]:
m.bas6.ibound[0].plot()

### Get the model spatial reference
We need to get information on the model grid so that `pyemu` can set up interpolation from pilot points to the model cells:

In [ ]:
sr = pyemu.helpers.SpatialReference.from_namfile(
        os.path.join(main_opt, mname),
        delr=m.dis.delr.array, delc=m.dis.delc.array)

We are going to use a `pyemu` helper function to setup pilot points at cell centers for active cells:

In [ ]:
#START OF EXAMPLE CELLS
#THIS IS EXAMPLE AND LATER INCLUDED IN THE LOOP, NO NEED TO RUN THIS CELL
# Create pilot points as a shapefile
# we want hk pilot points in the top layer...
parName= "hk0"
prefix_dict = {0:[f"{parName}"]}
df_pp = pyemu.pp_utils.setup_pilotpoints_grid(ml=m,
                                              prefix_dict=prefix_dict,
                                              pp_dir=main_opt,
                                              tpl_dir=main_opt,
                                              every_n_cell=20,
                                              shapename=f'pp_{parName}.shp')
pp_file = os.path.join(main_opt,f"{parName}pp.dat")
# assert os.path.exists(pp_file)

In [ ]:
df_pp

So cool, we now defined pilot points as a set of spatially distributed parameters...but how do go from pilot points to the model input HK array? Answer: geostatistics.  

We need to calculate the geostatistical factors (weights) used to form the interpolated value for the HK value at each model cell - its a spatially-weighted combination of pilot point values

If you are not familiar or are rusty on geostatistics, consider checking out the [`intro_to_geostatistics`](https://github.com/gmdsi/GMDSI_notebooks/blob/main/tutorials/part0_intro_to_geostatistics/intro_to_geostatistics.ipynb) notebook 

## Need to create Kriging factors and regularization inputs
Following the guidelines in _Approaches to Highly Parameterized Inversion: Pilot-Point Theory, Guidelines, and Research Directions_ https://pubs.usgs.gov/sir/2010/5168/

### First we need to define a couple geostatistical structures (e.g. variograms)

From _PEST Groundwater Data Utilities Part A: Overview_ page 43, there are 4 acceptable variogram types:

 1. *Spherical*  
### $\gamma\left(h\right)=c\times\left[1.5\frac{h}{a}-0.5\frac{h}{a}^3\right]$ if $h<a$
### $\gamma\left(h\right)=c$ if $h \ge a$  
     
 2. *Exponential*  
### $\gamma\left(h\right)=c\times\left[1-\exp\left(-\frac{h}{a}\right)\right]$  
     
 3. *Gaussian*  
### $\gamma\left(h\right)=c\times\left[1-\exp\left(-\frac{h^2}{a^2}\right)\right]$  
 
 4. *Power*  
### $\gamma\left(h\right)=c\times h^a$
     
 The number refers to `VARTYPE`. 
 
 `BEARING` and `ANISOTROPY` only apply if there is a principal direction of anisotropy. 
 
 $h$ is the separation distance, and $a$ is the range, expressed with the `A` parameter.


### First, let's create ``variogram`` and ``GeoStruct`` objects.  

These describe how HK varies spatailly, remember?

In [ ]:
pyemu.geostats.GeoStruct?

In [ ]:
v = pyemu.geostats.ExpVario(contribution=0.7,a=20000)
gs = pyemu.geostats.GeoStruct(variograms=v,nugget=0.0, transform="log")
ax = gs.plot()
ax.grid()
# ax.set_ylim(0,2.0)

Now, let's get an ``OrdinaryKrige`` object, which needs the ``GeoStruct`` as well as the x, y, and name of the pilot point locations (which happens to be in that really cool ``df_pp`` instance)

In [ ]:
ok = pyemu.geostats.OrdinaryKrige(gs,df_pp)

In [ ]:
os.getcwd()

Once the ``OrdinaryKrige`` object is created, we need to calculate the geostatistical interpolation factors for each model cell.  We do this with the ``.calc_factors_grid()`` method: it needs to know about the model's spatial orientation and also accepts some optional arguments:

In [ ]:
#NO NEED TO RUN THIS CELL, THIS IS JUST EXAMPLE, IT IS INCLUDED LATER IN THE LOOP
df = ok.calc_factors_grid(
            sr,
            var_filename= f"{parName}pp.var.ref",
            minpts_interp=1,
            maxpts_interp=10,
            search_radius=200000,
            verbose=True,
            num_threads=12
)

One of the really cool things about geostatistics is that it gives you both the interpolation (factors), but also gives you the uncertainty in the areas between control (pilot) points.  Above, we wrote this uncertainty information to an array that has the same rows and cols as the model grid - this array is very useful for understanding the function of the variogram.

In [ ]:
#NO NEED TO RUN THIS CELL, THIS IS JUST EXAMPLE, IT IS INCLUDED LATER IN THE LOOP
# arr_var = np.loadtxt(pst_name.replace(".pst",".var.ref"))
arr_var = np.loadtxt(f"{parName}pp.var.ref")
ax = plt.subplot(111,aspect="equal")
p = ax.imshow(arr_var,extent=sr.get_extent(),alpha=0.25)
plt.colorbar(p)
ax.scatter(df_pp.x,df_pp.y,marker='.',s=4,color='r')
#END OF EXAMPLE CELLS

We see that at the pilot point locations (red dots), the uncertainty in the geostats is minimal...as expected. The color scale is uncertainty. It increases with distance to pilot points.

The call to ``.calc_factors_grid()`` also returns a ``DataFrame`` which has useful info - lets look:

In [ ]:
df.head()

We see that there is one row for each model cell, and for each row, we see the distance, names, and weight for the "nearby" pilot points.  The interpolated value for cells that have a pilot point at their center only need one weight - 1.0 - and one pilot point.  Other cells are weighted combinations of pilot points. 

### Back to linking pilot points to grid values

Now we need to save the factors (weights) to a special file that we will use later to quickly generate a new HK array from a set of pilot point values:

In [ ]:
ok.to_grid_factors_file(pp_file+".fac")

Just for demo purposes, let's generate ``random`` pilot point values and run them through the factors to see what the ``hk`` array looks like

In [ ]:
# generate random values

df_pp.loc[:,"parval1"] = np.random.randint(0.1, 100, size=df_pp.shape[0])
# fixedvals = [1, 10, 10, 90, 90, 20, 50, 30, 70]
# df_pp.loc[:,"parval1"] = fixedvals
# save a pilot points file
pyemu.pp_utils.write_pp_file(pp_file,df_pp)

In [ ]:
# interpolate the pilot point values to the grid
hk_arr = pyemu.geostats.fac2real(pp_file, factors_file=pp_file+".fac",out_file=None, )

In [ ]:
# plot
ax = plt.subplot(111,aspect='equal')
p = ax.imshow(hk_arr,interpolation="nearest",extent=sr.get_extent(),alpha=0.5)
plt.colorbar(p)
ax.scatter(df_pp.x,df_pp.y,marker='.',s=4,color='k')


What happens if you recalculate the factors using one point for every cell? Change ``maxpts_interp`` to 1 in the ``calc_factors_grid()`` and rerun these cells.

### (Foreshadowing) An aside on geostatistics and covariance matrices

The ``GeoStruct`` object above was used to interpolate from pilot point locations to each node in the grid.  But this same ``GoeStruct`` also has important information regarding how the pilot points are related to each other spatially---that is, the ``GeoStruct`` object implies a covariance matrix.  Let's form that matrix 

In [ ]:
cov = gs.covariance_matrix(df_pp.x,df_pp.y,df_pp.parnme)

In [ ]:
plt.imshow(cov.x)
plt.colorbar()

In [ ]:
cov.to_dataframe().head()

What do these numbers mean?  Why should you care?  Well, this covariance matrix plays an important role in uncertainty quantification, as well as in governing the way pilot point parameters are adjusted during calibration. We will return to these topics in future tutorials.

 Now back to setting up our pilot points and control file.

### Build a control file using these pilot points. Let's automate the process for HK, SY, and SS.

In [ ]:
from swatmf.utils import pst_utils

In [ ]:
pst_utils.create_pp_tpl_files?

In [ ]:
pst_utils.create_pp_tpl_files(
    main_opt, mname,
    parNams=["hk0", "sy0", "ss0"],
    contributions=[0.7, 0.5, 0.4],
    cellNums=20,
    ranges=20000,
    nuggets=0
)

## 2.3 SWAT parameters with SWAT pars database file

In [ ]:
from swatmf.utils.swat_configs import SwatEdit

In [ ]:
os.chdir(main_opt)

In [ ]:
m1 = SwatEdit(os.getcwd())

In [ ]:
os.getcwd()

In [ ]:
m1.read_swat_pars_db()

In [ ]:
m1.create_swat_pars_cal()

# 3. Build instruction files
### Let's do initial run!

In [ ]:
pyemu.os_utils.run('swatmf_rel230922.exe', cwd='.')

## 3.1 Depth to watertable (MODFLOW) 

In [ ]:
swatmf_pst_utils.extract_depth_to_water(grids, sim_start, cal_end)

In [ ]:
mf_obs_grid_ids = pd.read_csv(
                    'modflow.obs',
                    sep=r'\s+',
                    usecols=[3, 4],
                    skiprows=2,
                    header=None
                    )
sim_grids = mf_obs_grid_ids.iloc[:, 0].tolist()

In [ ]:
sim_grids

In [ ]:
swatmf_pst_utils.extract_depth_to_water(sim_grids, sim_start, cal_end)

### match it with modflow.obd file (MODFLOW)

In [ ]:
swatmf_pst_utils.mf_obd_to_ins('dtw_2801.txt', 'gid2801', cal_start, cal_end)

## 4.1 Streamflow (SWAT)

In [ ]:
# extract daily stream discharge
swatmf_pst_utils.extract_month_stf(subs, sim_start, warmup, cal_start, cal_end)

In [ ]:
swatmf_pst_utils.mf_obd_to_ins('dtw_2801.txt', 'gid2801', cal_start, cal_end)

## 4.2 match it with dtw_obd file (MODFLOW)

In [ ]:

swatmf_pst_utils.stf_obd_to_ins?

In [ ]:
swatmf_pst_utils.stf_obd_to_ins('str_037.txt', 'sub_37',cal_start, cal_end, time_step='month')

# 5. Create PEST control file

In [ ]:
io_files = pyemu.helpers.parse_dir_for_io_files('.')
pst = pyemu.Pst.from_io_files(*io_files)
pyemu.helpers.pst_from_io_files(io_files[0], io_files[1], io_files[2], io_files[3], 'uml_dummy.pst')
io_files

In [ ]:
par = pst.parameter_data
par

## 5.1 Assign parameter group name

In [ ]:
for i in range(len(par)):
    if (par.iloc[i, 0][:2]) == 'sy':
        par.iloc[i, 6] = 'sy'
    elif par.iloc[i, 0][:7] == 'rivbot_':
        par.iloc[i, 6] = 'rivbot'
    elif par.iloc[i, 0][:6] == 'rivcd_':
        par.iloc[i, 6] = 'rivcd'
    elif par.iloc[i, 0][:2] == 'hk':
        par.iloc[i, 6] = 'hk'
    elif par.iloc[i, 0][:2] == 'ss':
        par.iloc[i, 6] = 'ss'
    else:
        par.iloc[i, 6] = 'swat'
print(par)

## 5.2 Adjust initial parameter values and their ranges

In [ ]:
count = 0
for i in range(len(par)):
    if (par.iloc[i, 6] == 'hk'):
        par.iloc[i, 3] = 18               #initial value
        par.iloc[i, 4] = 4.200000e-01    #lower value
        par.iloc[i, 5] = 4.050000e+02    #upper value
    elif (par.iloc[i, 6] == 'sy'):
        par.iloc[i, 3] = 2.000000e-01 #intial value       
        par.iloc[i, 4] = 1.000000e-04 #lower value
        par.iloc[i, 5] = 0.6          #upper value
    elif (par.iloc[i, 6] == 'ss'):
        par.iloc[i, 3] = 3.300000e-06 #inital value       
        par.iloc[i, 4] = 3.000000e-06 #lower value
        par.iloc[i, 5] = 3.400000e-06 #upper value
    elif (par.iloc[i, 6] == 'rivbot'):
        par.iloc[i, 3] = 3.001     
        par.iloc[i, 4] = 0.001
        par.iloc[i, 5] = 6
        par.iloc[i, 8] = -3
    elif (par.iloc[i, 6] == 'rivcd'):
        par.iloc[i, 3] = 50.001       
        par.iloc[i, 4] = 0.001
        par.iloc[i, 5] = 100
        par.iloc[i, 8] = -50
    else:
        count += 1
count

In [ ]:
# CN2
par.loc['cn2', 'parval1'] = 66
par.loc['cn2', 'parlbnd'] = 35
par.loc['cn2', 'parubnd'] = 92
par.loc['cn2', 'offset'] = -1

# sol_k()
par.loc['sol_k()', 'parval1'] = 12
par.loc['sol_k()', 'parlbnd'] = 0.7
par.loc['sol_k()', 'parubnd'] = 57
par.loc['sol_k()', 'offset'] = -1

# sol_awc()
par.loc['sol_awc()', 'parval1'] = 1.001
par.loc['sol_awc()', 'parlbnd'] = 0.5
par.loc['sol_awc()', 'parubnd'] = 1.5
par.loc['sol_awc()', 'offset'] = -1

# ESCO
par.loc['esco', 'parval1'] = 1.001
par.loc['esco', 'parlbnd'] = 0.5
par.loc['esco', 'parubnd'] = 1.5
par.loc['esco', 'offset'] = -1


In [ ]:
par

## 5.3 Assign parameter group name

In [ ]:
# set observation group
obd = pst.observation_data
obd

In [ ]:
# Change obd group name
for i in range(len(obd)):
    if obd.iloc[i, 0][:3] == 'sub':
        obd.iloc[i, 3] = obd.iloc[i, 0][:-7]
    else:
        obd.iloc[i, 3] = obd.iloc[i, 0][:-9]
obd

## 5.4 Provide actual observed values to control file

In [ ]:
# Streamflow
stf_obd = pd.read_csv('stf_mon.obd',
                       sep='\t',
                       index_col = 0,
                       parse_dates = True,
                       na_values=[-999, '']
                     )
stf_obd = stf_obd[cal_start: cal_end]
stf_obd

In [ ]:
# watertable
dtw_obd = pd.read_csv('modflow.obd',
                       sep='\t',
                       index_col = 0,
                       parse_dates = True,
                       na_values=[-999, '']
                     )
dtw_obd = dtw_obd[cal_start: cal_end]
dtw_obd

In [ ]:
# Get sub list based on obd order
obd_order = []
for i in obd.obgnme.tolist():
    if i not in obd_order:
        obd_order.append(i)
obd_order

In [ ]:
stf_obd

In [ ]:
dtw_obd

In [ ]:
# get total list from each sub obd, delete na vals
tot_obd = []
for i in obd_order[:1]:
    tot_obd += stf_obd[i].dropna().tolist()
    print(i)
for i in obd_order[1:]:
    tot_obd += dtw_obd[i].dropna().tolist()
    print(i)

len(tot_obd)

In [ ]:
obd.loc[:, 'obsval'] = tot_obd
obd

# 6. Create new control file with settings

In [ ]:
pst.control_data.noptmax=0
pst.model_command = 'python forward_run.py'
pst.write('uml_pest.pst')